# Финальный анализ privacy attacks

Ноутбук собирает итоговые графики и таблицы по `MIA` и `data extraction`, сохраняет их в `artifacts/analysis/` и при необходимости логирует в `Comet`.

## 0. Setup

In [ ]:
!pip install -q -r requirements.txt
!pip install -q accelerate Pillow pandas tqdm einops timm comet_ml huggingface_hub
!pip install -q peft bitsandbytes trl qwen-vl-utils
%pip install -q seaborn matplotlib python-dotenv
!pip uninstall -y transformers tokenizers
!pip install -q tokenizers==0.21.0
!pip install -q transformers==4.49.0 --force-reinstall --no-deps


In [ ]:
import os



In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import comet_ml

import transformers
print(f"transformers version: {transformers.__version__}")
if not transformers.__version__.startswith("4.4"):
    print(f"WARNING: expected 4.4x.x, got {transformers.__version__}. Florence-2 may fail.")


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    return None


def get_colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.getenv(name)


IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/course_work2026')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    repo_dir = Path('/content/course_work2026')
    if not repo_dir.exists():
        subprocess.run([
            'git', 'clone',
            get_colab_secret('COURSE_WORK2026_REPO_URL') or 'https://github.com/sk3feel/hidden-data-reproduction-multimodal.git',
            str(repo_dir),
        ], check=True)
    PROJECT_ROOT = find_project_root(repo_dir) or find_project_root(Path.cwd())
    if PROJECT_ROOT is None:
        raise RuntimeError('Could not locate project root in Colab even after cloning the repository.')
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    if PROJECT_ROOT is None:
        raise RuntimeError('Could not locate project root. Open the notebook from the repository workspace or set the working directory inside the repo.')
    DRIVE_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

if IS_COLAB:
    from src.colab_setup import setup_colab
    setup_summary, bootstrap_experiment = setup_colab(repo_url=get_colab_secret('COURSE_WORK2026_REPO_URL') or 'https://github.com/sk3feel/hidden-data-reproduction-multimodal.git')
    bootstrap_experiment.set_name('final-analysis-bootstrap')
    bootstrap_experiment.end()
    PROJECT_ROOT = Path(setup_summary['project_root'])
    sys.path.insert(0, str(PROJECT_ROOT))
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

DATA_ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts'
ARTIFACTS_ROOT = DRIVE_ROOT / 'artifacts_v2_controlled'
MIA_DIR = ARTIFACTS_ROOT / 'privacy_attacks' / 'mia'
EXTRACTION_DIR = ARTIFACTS_ROOT / 'privacy_attacks' / 'extraction'
ANALYSIS_DIR = ARTIFACTS_ROOT / 'analysis'
FIGURES_DIR = ANALYSIS_DIR / 'figures'
TABLES_DIR = ANALYSIS_DIR / 'tables'
QUAL_DIR = ANALYSIS_DIR / 'qualitative'
for path in [ANALYSIS_DIR, FIGURES_DIR, TABLES_DIR, QUAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FINAL_EXPERIMENT = comet_ml.Experiment(project_name=os.getenv('COMET_PROJECT_NAME') or 'qwen3-1', workspace=os.getenv('COMET_WORKSPACE') or 'scfeel')
FINAL_EXPERIMENT.set_name('final-analysis-v2')

print({'project_root': str(PROJECT_ROOT), 'drive_root': str(DRIVE_ROOT), 'mia_dir': str(MIA_DIR), 'extraction_dir': str(EXTRACTION_DIR), 'analysis_dir': str(ANALYSIS_DIR), 'is_colab': IS_COLAB})

In [ ]:
from __future__ import annotations

import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.inference_scenarios import generate_scenario_image


sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 12,
    'axes.titlesize': 12,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
})

MODEL_LABELS = {
    'florence2': 'Florence-2-base',
    'qwen2b': 'Qwen2-VL-2B',
    'qwen7b': 'Qwen2-VL-7B',
}
TRAIN_LOSS_LABELS = {
    'losses_run1': 'Florence-2',
    'losses_qwen2b': 'Qwen2-VL-2B',
    'losses_qwen7b': 'Qwen2-VL-7B',
}
MODEL_PARAMS = {
    'florence2': 230_000_000,
    'qwen2b': 2_000_000_000,
    'qwen7b': 7_000_000_000,
}
IMAGE_SCENARIO_ORDER = ['original', 'img_black', 'img_blur_20', 'img_blur_50']
IMAGE_OCR_SCENARIO_ORDER = [
    'img_black__ocr_mask__k_0',
    'img_none__ocr_mask__k_0',
]
QWEN_OCR_SCENARIO_MAP = {
    'img_black__ocr_mask__k_0': 'ocr_mask__img_black__k_0',
    'img_none__ocr_mask__k_0': 'ocr_mask__img_none__k_0',
}
COMET_DOWNLOAD_EXPERIMENT_KEYS = []
COMET_FALLBACK_EXPERIMENT_NAMES = {
    'mia-baseline-summary',
    'mia-finetuned-summary',
    'mia-epoch-curve',
    'florence2-mia-baseline',
    'qwen2b-mia-baseline',
    'qwen7b-mia-baseline',
    'florence2-mia-finetuned',
    'qwen2b-mia-finetuned',
    'qwen7b-mia-finetuned',
    'extraction-baseline',
    'florence2-extraction',
    'qwen2b-extraction-imageonly',
    'qwen2b-extraction-imageocr',
    'qwen7b-extraction-imageonly',
    'qwen7b-extraction-imageocr',
    'extraction-epoch-curve',
}


def save_figure(fig, stem: str):
    png_path = FIGURES_DIR / f'{stem}.png'
    pdf_path = FIGURES_DIR / f'{stem}.pdf'
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(pdf_path, dpi=300, bbox_inches='tight')
    FINAL_EXPERIMENT.log_figure(stem, fig)
    FINAL_EXPERIMENT.log_asset(str(png_path), file_name=png_path.name)
    FINAL_EXPERIMENT.log_asset(str(pdf_path), file_name=pdf_path.name)
    return png_path, pdf_path


def save_table(df: pd.DataFrame, stem: str):
    csv_path = TABLES_DIR / f'{stem}.csv'
    tex_path = TABLES_DIR / f'{stem}.tex'
    df.to_csv(csv_path, index=False)
    df.to_latex(tex_path, index=False, float_format=lambda x: f'{x:.3f}')
    FINAL_EXPERIMENT.log_table(csv_path.name, df)
    FINAL_EXPERIMENT.log_asset(str(csv_path), file_name=csv_path.name)
    FINAL_EXPERIMENT.log_asset(str(tex_path), file_name=tex_path.name)
    return csv_path, tex_path


def iter_comet_experiments_for_download(api, workspace: str, project_name: str, experiment_keys: list[str]):
    if experiment_keys:
        for experiment_key in experiment_keys:
            try:
                yield api.get_experiment_by_key(experiment_key)
            except Exception as experiment_error:
                print({'experiment_key': experiment_key, 'error': str(experiment_error)})
        return

    try:
        experiments = api.get_experiments(workspace, project_name)
    except TypeError:
        experiments = api.get_experiments(workspace, project_name=project_name)

    for api_experiment in experiments:
        name = None
        if hasattr(api_experiment, 'get_name'):
            try:
                name = api_experiment.get_name()
            except Exception:
                name = None
        name = name or getattr(api_experiment, 'name', None)
        if name in COMET_FALLBACK_EXPERIMENT_NAMES:
            yield api_experiment


def maybe_download_csvs_from_comet(target_dir: Path, experiment_keys: list[str]):
    if any(target_dir.glob('*.csv')) or not experiment_keys:
        if any(target_dir.glob('*.csv')):
            return
    api_key = os.getenv('COMET_API_KEY') or get_colab_secret('COMET_API_KEY')
    if not api_key:
        print(f'No local CSV in {target_dir} and COMET_API_KEY is not available; skipping Comet download.')
        return
    workspace = os.getenv('COMET_WORKSPACE') or get_colab_secret('COMET_WORKSPACE') or 'scfeel'
    project_name = os.getenv('COMET_PROJECT_NAME') or get_colab_secret('COMET_PROJECT_NAME') or 'qwen3-1'
    api = comet_ml.API(api_key=api_key)
    for api_experiment in iter_comet_experiments_for_download(api, workspace, project_name, experiment_keys):
        experiment_key = getattr(api_experiment, 'key', None)
        experiment_name = getattr(api_experiment, 'name', None)
        try:
            asset_list = api_experiment.get_asset_list()
        except Exception as asset_list_error:
            print({'experiment_key': experiment_key, 'experiment_name': experiment_name, 'error': str(asset_list_error)})
            continue
        for asset in asset_list:
            file_name = asset.get('fileName') or asset.get('file_name') or asset.get('title')
            asset_id = asset.get('assetId') or asset.get('asset_id')
            if not file_name or not file_name.endswith('.csv') or not asset_id:
                continue
            destination = target_dir / file_name
            if destination.exists():
                continue
            try:
                payload = api_experiment.get_asset(asset_id, return_type='binary', stream=False)
                destination.write_bytes(payload)
            except Exception as asset_error:
                print({'experiment_key': experiment_key, 'experiment_name': experiment_name, 'file_name': file_name, 'error': str(asset_error)})


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing required CSV: {path}. Put results into artifacts/privacy_attacks/ or make COMET_API_KEY available for fallback download.')
    return pd.read_csv(path)


def load_optional_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def build_mia_score_df(path: Path) -> pd.DataFrame:
    df = load_csv(path).copy()
    if 'is_seen' not in df.columns and 'split' in df.columns:
        df['is_seen'] = (df['split'] == 'seen').astype(int)
    if 'coarse_type' not in df.columns:
        if 'field_type' in df.columns:
            df['coarse_type'] = df['field_type']
        else:
            df['coarse_type'] = 'OTHER'
    if 'question_id' not in df.columns:
        if 'example_id' in df.columns:
            df['question_id'] = df['example_id'].astype(str)
        else:
            df['question_id'] = [str(i) for i in range(len(df))]
    return df


def build_mia_coarse_dfs(field_auc_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    tag_map = {
        'florence2_epoch10': 'florence2',
        'qwen2b_epoch10': 'qwen2b',
        'qwen7b_epoch10': 'qwen7b',
    }
    result = {}
    for source_tag, target_tag in tag_map.items():
        subset = field_auc_df[field_auc_df['model'] == source_tag].copy()
        if subset.empty:
            result[target_tag] = pd.DataFrame(columns=['coarse_type', 'auc_confidence', 'auc_loss', 'n_examples'])
            continue
        subset['coarse_type'] = subset['field_type']
        result[target_tag] = subset
    return result


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as fh:
        for line in fh:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def load_records(manifest_path: Path, split: str, limit: int | None = None) -> list[dict]:
    records = load_jsonl(manifest_path)
    if limit is not None:
        records = records[:limit]
    image_dir = manifest_path.parent / 'images' / 'original'
    normalized = []
    for idx, record in enumerate(records):
        image_name = str(record['image_path']).replace('\\', '/').split('/')[-1]
        updated = dict(record)
        updated['image_path'] = str(image_dir / image_name)
        updated['split'] = split
        updated['record_key'] = f"{split}:{record.get('example_id', 'na')}:{record.get('local_row_id', idx)}"
        normalized.append(updated)
    return normalized


maybe_download_csvs_from_comet(MIA_DIR, COMET_DOWNLOAD_EXPERIMENT_KEYS)
maybe_download_csvs_from_comet(EXTRACTION_DIR, COMET_DOWNLOAD_EXPERIMENT_KEYS)

TRAIN_RECORDS = load_records(DATA_ARTIFACTS_ROOT / 'docqa_recovery' / 'benchmark_train' / 'manifest.jsonl', split='seen', limit=800)
VALIDATION_RECORDS = load_records(DATA_ARTIFACTS_ROOT / 'docqa_recovery' / 'benchmark' / 'manifest.jsonl', split='unseen', limit=800)
RECORD_LOOKUP = {record['record_key']: record for record in TRAIN_RECORDS + VALIDATION_RECORDS}
RECORD_LOOKUP_BY_ID_SPLIT = {(str(record.get('example_id', '')), record['split']): record for record in TRAIN_RECORDS + VALIDATION_RECORDS}

MIA_BASELINE_SUMMARY = load_csv(MIA_DIR / 'baseline_mia_summary.csv')
MIA_FINETUNED_SUMMARY = load_csv(MIA_DIR / 'finetuned_mia_summary.csv')
MIA_EPOCH_CURVE = load_optional_csv(MIA_DIR / 'mia_epoch_curve.csv')
MIA_SCORE_DFS = {
    'florence2': build_mia_score_df(MIA_DIR / 'florence2_epoch10.csv'),
    'qwen2b': build_mia_score_df(MIA_DIR / 'qwen2b_epoch10.csv'),
    'qwen7b': build_mia_score_df(MIA_DIR / 'qwen7b_epoch10.csv'),
}
FIELD_TYPE_AUC_DF = load_csv(MIA_DIR / 'field_type_auc.csv')
MIA_COARSE_DFS = build_mia_coarse_dfs(FIELD_TYPE_AUC_DF)

EXTRACTION_BASELINE = load_csv(EXTRACTION_DIR / 'baseline_florence2_image_only_metrics.csv')
EXTRACTION_EPOCH_CURVE = load_optional_csv(EXTRACTION_DIR / 'extraction_epoch_curve.csv')
EXTRACTION_METRICS = {
    'florence2_image_only': load_csv(EXTRACTION_DIR / 'florence2_epoch10_image_only_metrics.csv'),
    'qwen2b_image_only': load_csv(EXTRACTION_DIR / 'qwen2b_epoch10_image_only_metrics.csv'),
    'qwen2b_image_ocr': load_csv(EXTRACTION_DIR / 'qwen2b_epoch10_image_ocr_metrics.csv'),
    'qwen7b_image_only': load_csv(EXTRACTION_DIR / 'qwen7b_epoch10_image_only_metrics.csv'),
    'qwen7b_image_ocr': load_optional_csv(EXTRACTION_DIR / 'qwen7b_epoch10_image_ocr_metrics.csv'),
}
EXTRACTION_SUCCESS = {
    'florence2_image_only': load_csv(EXTRACTION_DIR / 'florence2_epoch10_image_only_successful_examples.csv'),
    'qwen2b_image_only': load_csv(EXTRACTION_DIR / 'qwen2b_epoch10_image_only_successful_examples.csv'),
    'qwen2b_image_ocr': load_csv(EXTRACTION_DIR / 'qwen2b_epoch10_image_ocr_successful_examples.csv'),
    'qwen7b_image_only': load_csv(EXTRACTION_DIR / 'qwen7b_epoch10_image_only_successful_examples.csv'),
    'qwen7b_image_ocr': load_optional_csv(EXTRACTION_DIR / 'qwen7b_epoch10_image_ocr_successful_examples.csv'),
}
TRAIN_LOSS_DIRS = [
    DRIVE_ROOT / 'artifacts' / 'finetuning_generative',
    DATA_ARTIFACTS_ROOT / 'finetuning_generative',
]
TRAIN_LOSS_DFS = {}
for stem in TRAIN_LOSS_LABELS:
    for candidate_dir in TRAIN_LOSS_DIRS:
        candidate = candidate_dir / f'{stem}.csv'
        if candidate.exists():
            TRAIN_LOSS_DFS[stem] = load_csv(candidate)
            break

print({'mia_csv_count': len(list(MIA_DIR.glob('*.csv'))), 'extraction_csv_count': len(list(EXTRACTION_DIR.glob('*.csv'))), 'train_loss_csv_count': len(TRAIN_LOSS_DFS)})

## 1. MIA графики

In [ ]:
mia_bar_rows = []
for _, row in MIA_BASELINE_SUMMARY.iterrows():
    mia_bar_rows.append({'model': row['tag'], 'stage': 'baseline', 'method': 'confidence', 'auc': row['auc_confidence']})
    mia_bar_rows.append({'model': row['tag'], 'stage': 'baseline', 'method': 'loss', 'auc': row['auc_loss']})
for _, row in MIA_FINETUNED_SUMMARY.iterrows():
    mia_bar_rows.append({'model': row['tag'], 'stage': 'fine-tuned', 'method': 'confidence', 'auc': row['auc_confidence']})
    mia_bar_rows.append({'model': row['tag'], 'stage': 'fine-tuned', 'method': 'loss', 'auc': row['auc_loss']})
mia_bar_df = pd.DataFrame(mia_bar_rows)
mia_bar_df['model_label'] = mia_bar_df['model'].map(MODEL_LABELS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, method in zip(axes, ['confidence', 'loss']):
    plot_df = mia_bar_df[mia_bar_df['method'] == method]
    sns.barplot(data=plot_df, x='model_label', y='auc', hue='stage', ax=ax, palette='Set2')
    ax.set_title('AUC-ROC по методу ' + ('confidence (средняя log-prob)' if method == 'confidence' else 'loss (cross-entropy)'))
    ax.set_xlabel('Модель')
    ax.set_ylabel('AUC-ROC')
    ax.axhline(0.5, linestyle='--', color='gray', linewidth=1)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
save_figure(fig, 'mia_bar_baseline_vs_finetuned')
display(mia_bar_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, tag in zip(axes, ['florence2', 'qwen2b', 'qwen7b']):
    df = MIA_SCORE_DFS[tag]
    ax.hist(df.loc[df['is_seen'] == 1, 'confidence'], bins=30, alpha=0.6, label='seen')
    ax.hist(df.loc[df['is_seen'] == 0, 'confidence'], bins=30, alpha=0.6, label='unseen')
    ax.set_title(MODEL_LABELS[tag])
    ax.set_xlabel('Средняя log-prob ответа')
    ax.set_ylabel('Частота')
    ax.legend()
plt.tight_layout()
save_figure(fig, 'mia_confidence_histograms')

In [ ]:
mia_heatmap_df = pd.concat([
    df.assign(tag=tag, model_label=MODEL_LABELS[tag])
    for tag, df in MIA_COARSE_DFS.items() if not df.empty
], ignore_index=True)
mia_heatmap = mia_heatmap_df.pivot(index='model_label', columns='coarse_type', values='auc_confidence')
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(mia_heatmap, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax)
ax.set_title('AUC-ROC MIA по типам полей')
ax.set_xlabel('Тип поля')
ax.set_ylabel('Модель')
plt.tight_layout()
save_figure(fig, 'mia_heatmap_model_fieldtype_auc')
display(mia_heatmap_df)

In [ ]:
epoch_curve_df = MIA_EPOCH_CURVE.copy()
if epoch_curve_df.empty:
    print('mia_epoch_curve.csv not found; skipping epoch curve plot.')
else:
    epoch_curve_df['model_label'] = epoch_curve_df['tag'].map(MODEL_LABELS)
    fig, ax = plt.subplots(figsize=(8, 5))
    for tag, group in epoch_curve_df.groupby('tag'):
        ax.plot(group['epoch'], group['auc_confidence'], marker='o', label=MODEL_LABELS[tag])
    ax.set_title('Кривая MIA по эпохам')
    ax.set_xlabel('Эпоха')
    ax.set_ylabel('AUC-ROC')
    ax.axhline(0.5, linestyle='--', color='gray', linewidth=1)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_figure(fig, 'mia_epoch_curve')
    display(epoch_curve_df)

if not TRAIN_LOSS_DFS:
    print('Training loss CSV files not found; skipping train loss plot.')
else:
    train_loss_rows = []
    for stem, df in TRAIN_LOSS_DFS.items():
        temp = df.copy()
        temp['source'] = stem
        temp['model_label'] = TRAIN_LOSS_LABELS[stem]
        train_loss_rows.append(temp)
    train_loss_df = pd.concat(train_loss_rows, ignore_index=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    for stem, group in train_loss_df.groupby('source'):
        group = group.sort_values('epoch')
        ax.plot(group['epoch'], group['loss'], marker='o', label=TRAIN_LOSS_LABELS[stem])
    ax.set_title('Кривая train loss по эпохам')
    ax.set_xlabel('Эпоха')
    ax.set_ylabel('Train loss')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_figure(fig, 'training_loss_curves')
    display(train_loss_df)

In [ ]:
scaling_df = MIA_FINETUNED_SUMMARY[['tag', 'auc_confidence', 'auc_loss']].copy()
scaling_df['params'] = scaling_df['tag'].map(MODEL_PARAMS)
scaling_df['log_params'] = np.log10(scaling_df['params'])
scaling_df['model_label'] = scaling_df['tag'].map(MODEL_LABELS)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(scaling_df['log_params'], scaling_df['auc_confidence'], marker='o', label='confidence')
ax.plot(scaling_df['log_params'], scaling_df['auc_loss'], marker='s', label='loss')
for _, row in scaling_df.iterrows():
    ax.annotate(row['model_label'], (row['log_params'], row['auc_confidence']), xytext=(4, 4), textcoords='offset points')
ax.set_title('Scaling MIA: log(параметры модели) -> AUC-ROC')
ax.set_xlabel('log10(число параметров)')
ax.set_ylabel('AUC-ROC')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, 'mia_scaling_auc')
display(scaling_df)

## 2. Extraction графики

In [ ]:
image_only_metrics = []
for key in ['florence2_image_only', 'qwen2b_image_only', 'qwen7b_image_only']:
    tag = key.split('_')[0]
    df = EXTRACTION_METRICS[key]
    subset = df[(df['split'] == 'seen') & (df['coarse_type'] == 'ALL')].copy()
    subset['tag'] = tag
    subset['model_label'] = MODEL_LABELS[tag]
    image_only_metrics.append(subset)
image_only_metrics_df = pd.concat(image_only_metrics, ignore_index=True)
heatmap_df = image_only_metrics_df.pivot(index='model_label', columns='scenario', values='corrected_em')[IMAGE_SCENARIO_ORDER]

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(heatmap_df, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax)
ax.set_title('Скорректированный EM по image-only сценариям (seen)')
ax.set_xlabel('Image-сценарий')
ax.set_ylabel('Модель')
plt.tight_layout()
save_figure(fig, 'extraction_heatmap_image_only_seen')
display(image_only_metrics_df)

In [ ]:
image_ocr_metrics = []
for key in ['qwen2b_image_ocr', 'qwen7b_image_ocr']:
    tag = key.split('_')[0]
    df = EXTRACTION_METRICS[key]
    if df.empty:
        continue
    subset = df[(df['split'] == 'seen') & (df['coarse_type'] == 'ALL')].copy()
    subset['tag'] = tag
    subset['model_label'] = MODEL_LABELS[tag]
    image_ocr_metrics.append(subset)
if not image_ocr_metrics:
    print('No image+OCR extraction metrics found; skipping OCR heatmap.')
else:
    image_ocr_metrics_df = pd.concat(image_ocr_metrics, ignore_index=True)
    heatmap_df = image_ocr_metrics_df.pivot(index='model_label', columns='scenario', values='corrected_em').reindex(columns=IMAGE_OCR_SCENARIO_ORDER)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.heatmap(heatmap_df, annot=True, fmt='.3f', cmap='YlOrBr', ax=ax)
    ax.set_title('Скорректированный EM для Qwen: image+OCR сценарии (seen)')
    ax.set_xlabel('Сценарий')
    ax.set_ylabel('Модель')
    plt.tight_layout()
    save_figure(fig, 'extraction_heatmap_image_ocr_seen')
    display(image_ocr_metrics_df)

In [ ]:
field_type_rows = []
for key in ['florence2_image_only', 'qwen2b_image_only', 'qwen7b_image_only']:
    tag = key.split('_')[0]
    df = EXTRACTION_METRICS[key]
    subset = df[(df['split'] == 'seen') & (df['scenario'] == 'img_black') & (df['coarse_type'] != 'ALL')].copy()
    subset['tag'] = tag
    subset['model_label'] = MODEL_LABELS[tag]
    field_type_rows.append(subset)
field_type_df = pd.concat(field_type_rows, ignore_index=True)

fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(data=field_type_df, x='coarse_type', y='corrected_em', hue='model_label', ax=ax)
ax.set_title('Скорректированный EM по типам полей, сценарий img_black (seen)')
ax.set_xlabel('Тип поля')
ax.set_ylabel('Скорректированный EM')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
save_figure(fig, 'extraction_bar_fieldtype_img_black')
display(field_type_df)

In [ ]:
blur_rows = []
for key in ['florence2_image_only', 'qwen2b_image_only', 'qwen7b_image_only']:
    tag = key.split('_')[0]
    df = EXTRACTION_METRICS[key]
    subset = df[(df['split'] == 'seen') & (df['coarse_type'] == 'ALL') & (df['scenario'].isin(['img_blur_20', 'img_blur_50']))].copy()
    subset['sigma'] = subset['scenario'].str.extract(r'(\d+)').astype(int)
    subset['tag'] = tag
    subset['model_label'] = MODEL_LABELS[tag]
    blur_rows.append(subset)
blur_df = pd.concat(blur_rows, ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 4))
for tag, group in blur_df.groupby('tag'):
    group = group.sort_values('sigma')
    ax.plot(group['sigma'], group['corrected_em'], marker='o', label=MODEL_LABELS[tag])
ax.set_title('Влияние blur sigma на скорректированный EM (seen)')
ax.set_xlabel('Sigma blur')
ax.set_ylabel('Скорректированный EM')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, 'extraction_blur_sigma_curve')
display(blur_df)

In [ ]:
mode_compare_rows = []
for tag in ['qwen2b', 'qwen7b']:
    image_only_df = EXTRACTION_METRICS[f'{tag}_image_only']
    image_ocr_df = EXTRACTION_METRICS[f'{tag}_image_ocr']
    mode_compare_rows.append({'tag': tag, 'model_label': MODEL_LABELS[tag], 'mode': 'image_only', 'mean_corrected_em': image_only_df[(image_only_df['split'] == 'seen') & (image_only_df['coarse_type'] == 'ALL')]['corrected_em'].mean()})
    if not image_ocr_df.empty:
        mode_compare_rows.append({'tag': tag, 'model_label': MODEL_LABELS[tag], 'mode': 'image+OCR', 'mean_corrected_em': image_ocr_df[(image_ocr_df['split'] == 'seen') & (image_ocr_df['coarse_type'] == 'ALL')]['corrected_em'].mean()})
mode_compare_df = pd.DataFrame(mode_compare_rows)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=mode_compare_df, x='model_label', y='mean_corrected_em', hue='mode', ax=ax, palette='Set2')
ax.set_title('Сравнение режимов: image-only vs image+OCR')
ax.set_xlabel('Модель')
ax.set_ylabel('Средний скорректированный EM')
plt.tight_layout()
save_figure(fig, 'extraction_mode_compare')
display(mode_compare_df)

In [ ]:
epoch_curve_df = EXTRACTION_EPOCH_CURVE.copy()
if epoch_curve_df.empty:
    print('extraction_epoch_curve.csv not found; skipping epoch curve plot.')
else:
    epoch_curve_df['model_label'] = epoch_curve_df['tag'].map(MODEL_LABELS)
    fig, ax = plt.subplots(figsize=(8, 5))
    for tag, group in epoch_curve_df.groupby('tag'):
        ax.plot(group['epoch'], group['corrected_em'], marker='o', label=MODEL_LABELS[tag])
    ax.set_title('Кривая extraction по эпохам (img_black, seen)')
    ax.set_xlabel('Эпоха')
    ax.set_ylabel('Скорректированный EM')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_figure(fig, 'extraction_epoch_curve')
    display(epoch_curve_df)

In [ ]:
scatter_rows = []
for tag in ['florence2', 'qwen2b', 'qwen7b']:
    mia_auc = float(MIA_FINETUNED_SUMMARY.loc[MIA_FINETUNED_SUMMARY['tag'] == tag, 'auc_loss'].iloc[0])
    extraction_em = float(EXTRACTION_METRICS[f'{tag}_image_only'].loc[(EXTRACTION_METRICS[f'{tag}_image_only']['split'] == 'seen') & (EXTRACTION_METRICS[f'{tag}_image_only']['scenario'] == 'img_black') & (EXTRACTION_METRICS[f'{tag}_image_only']['coarse_type'] == 'ALL'), 'corrected_em'].iloc[0])
    scatter_rows.append({'tag': tag, 'model_label': MODEL_LABELS[tag], 'mia_auc': mia_auc, 'extraction_corrected_em': extraction_em})
scatter_df = pd.DataFrame(scatter_rows)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(scatter_df['mia_auc'], scatter_df['extraction_corrected_em'], s=90)
for _, row in scatter_df.iterrows():
    ax.annotate(row['model_label'], (row['mia_auc'], row['extraction_corrected_em']), xytext=(5, 5), textcoords='offset points')
ax.set_title('Связь MIA и extraction')
ax.set_xlabel('MIA AUC-ROC по loss')
ax.set_ylabel('Extraction скорректированный EM (img_black, seen)')
ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, 'mia_vs_extraction_scatter')
display(scatter_df)

## 3. Качественный анализ

In [ ]:
success_examples_df = pd.concat([
    df.assign(source=name)
    for name, df in EXTRACTION_SUCCESS.items() if not df.empty
], ignore_index=True).head(10).copy()
save_table(success_examples_df, 'table_qual_success_examples')
display(success_examples_df)

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(12, 18))
axes = axes.flatten()
for ax, (_, row) in zip(axes, success_examples_df.iterrows()):
    record = None
    if 'record_key' in row and pd.notna(row['record_key']):
        record = RECORD_LOOKUP.get(row['record_key'])
    if record is None:
        record = RECORD_LOOKUP_BY_ID_SPLIT.get((str(row.get('example_id', '')), row.get('split', '')))
    if record is None:
        ax.axis('off')
        missing_title = f"{row['source']} | {row['scenario']}\nRecord not found for example_id={row.get('example_id', 'na')}"
        ax.set_title(textwrap.fill(missing_title, 45), fontsize=10)
        continue
    scenario = QWEN_OCR_SCENARIO_MAP.get(row['scenario'], row['scenario'])
    image = generate_scenario_image(record, scenario)
    ax.imshow(image)
    ax.axis('off')
    gold_value = row['answer'] if 'answer' in row else row.get('gold_answers', '')
    title = f"{row['source']} | {row['scenario']}\nЭталон: {str(gold_value)[:40]}\nОтвет: {str(row['prediction'])[:40]}"
    ax.set_title(textwrap.fill(title, 45), fontsize=10)
for ax in axes[len(success_examples_df):]:
    ax.axis('off')
plt.tight_layout()
save_figure(fig, 'qualitative_successful_extractions')

In [ ]:
gap_rows = []
for tag, df in MIA_SCORE_DFS.items():
    unseen_mean = df.loc[df['is_seen'] == 0, 'confidence'].mean()
    seen_df = df.loc[df['is_seen'] == 1].copy()
    seen_df['confidence_gap'] = seen_df['confidence'] - unseen_mean
    seen_df['tag'] = tag
    gap_rows.append(seen_df)
confidence_gap_df = pd.concat(gap_rows, ignore_index=True).sort_values('confidence_gap', ascending=False).head(5).copy()
confidence_gap_df['model_label'] = confidence_gap_df['tag'].map(MODEL_LABELS)
save_table(confidence_gap_df, 'table_qual_confidence_gap_examples')
display(confidence_gap_df[['model_label', 'question_id', 'coarse_type', 'confidence', 'confidence_gap', 'loss']])

In [ ]:
patterns = []
best_mia = MIA_FINETUNED_SUMMARY.sort_values('auc_confidence', ascending=False).iloc[0]
patterns.append(f"Максимальный MIA риск по confidence показывает {MODEL_LABELS[best_mia['tag']]}: AUC={best_mia['auc_confidence']:.3f}.")
best_extraction = scatter_df.sort_values('extraction_corrected_em', ascending=False).iloc[0]
patterns.append(f"Наилучший extraction в сценарии img_black показывает {best_extraction['model_label']}: corrected EM={best_extraction['extraction_corrected_em']:.3f}.")
best_field = field_type_df.groupby('coarse_type', as_index=False)['corrected_em'].mean().sort_values('corrected_em', ascending=False).iloc[0]
patterns.append(f"Самый уязвимый coarse type в среднем по моделям на img_black: {best_field['coarse_type']} (corrected EM={best_field['corrected_em']:.3f}).")
best_ocr = image_ocr_metrics_df.groupby('scenario', as_index=False)['corrected_em'].mean().sort_values('corrected_em', ascending=False).iloc[0]
patterns.append(f"Самый сильный image+OCR сценарий в среднем по Qwen: {best_ocr['scenario']} (corrected EM={best_ocr['corrected_em']:.3f}).")
patterns_df = pd.DataFrame({'pattern': patterns})
save_table(patterns_df, 'table_qual_patterns')
display(patterns_df)

## 4. Сводные таблицы

In [ ]:
table1 = mia_bar_df.pivot_table(index=['model_label', 'stage'], columns='method', values='auc').reset_index()
table2 = mia_heatmap_df[['model_label', 'coarse_type', 'auc_confidence', 'auc_loss', 'n_examples']].sort_values(['model_label', 'coarse_type']).reset_index(drop=True)
table3 = image_only_metrics_df[['model_label', 'split', 'scenario', 'corrected_em', 'corrected_f1', 'exact_match', 'token_f1']].sort_values(['model_label', 'split', 'scenario']).reset_index(drop=True)
table4 = image_ocr_metrics_df[['model_label', 'split', 'scenario', 'corrected_em', 'corrected_f1', 'exact_match', 'token_f1']].sort_values(['model_label', 'split', 'scenario']).reset_index(drop=True)
table5 = field_type_df[['model_label', 'coarse_type', 'corrected_em', 'corrected_f1', 'exact_match', 'token_f1']].sort_values(['coarse_type', 'model_label']).reset_index(drop=True)
table6 = scatter_df.merge(scaling_df[['model_label', 'params']], on='model_label', how='left').sort_values('params').reset_index(drop=True)

for idx, df in enumerate([table1, table2, table3, table4, table5, table6], start=1):
    save_table(df, f'table_{idx:02d}')

display(table1)
display(table2.head())
display(table3.head())
display(table4.head())
display(table5.head())
display(table6)

In [ ]:
analysis_paths = sorted([str(path) for path in FIGURES_DIR.glob('*')] + [str(path) for path in TABLES_DIR.glob('*')])
analysis_manifest = pd.DataFrame({'artifact_path': analysis_paths})
analysis_manifest_csv = ANALYSIS_DIR / 'analysis_manifest.csv'
analysis_manifest.to_csv(analysis_manifest_csv, index=False)
FINAL_EXPERIMENT.log_asset(str(analysis_manifest_csv), file_name=analysis_manifest_csv.name)
display(analysis_manifest)
FINAL_EXPERIMENT.end()